In [1]:
import meshio
import numpy as np

In [16]:
import meshio
import numpy as np
import tempfile, pathlib

data_dir = pathlib.Path("/Users/christian/Nextcloud/01_projects/Thermosim/ref_01_2mm")

# ── FEniCSx: time-series XDMF — read last timestep ──────────────────────────
with meshio.xdmf.TimeSeriesReader(data_dir / "ref_01_2mm_001.xdmf") as reader:
    points_fenics, cells_fenics = reader.read_points_cells()
    n_steps = reader.num_steps
    t, point_data, _ = reader.read_data(n_steps -1 )

T_fenics = point_data["T"]
print(f"FEniCSx : {n_steps} steps, last t={t:.2f}s, nodes={len(T_fenics)}")

# ── Abaqus VTU: fix comma-version quirk before parsing ───────────────────────
vtu_raw = (data_dir / "Step-1_-1.vtu").read_text()
vtu_fixed = vtu_raw.replace('version="1,0"', 'version="1.0"')

with tempfile.NamedTemporaryFile(suffix=".vtu", mode="w", delete=False) as f:
    f.write(vtu_fixed)
    tmp_path = f.name

m2 = meshio.read(tmp_path)
pathlib.Path(tmp_path).unlink()

print(f"Abaqus  : nodes={len(m2.points)}, fields={list(m2.point_data.keys())}")

FEniCSx : 101 steps, last t=100.00s, nodes=111450
Abaqus  : nodes=111450, fields=['NT11', 'RFL11']


In [17]:
from scipy.interpolate import LinearNDInterpolator

T_abaqus_raw = m2.point_data["NT11"]

print(f"FEniCSx nodes : {points_fenics.shape}  T range: {T_fenics.min():.4f} – {T_fenics.max():.4f}")
print(f"Abaqus  nodes : {m2.points.shape}       T range: {T_abaqus_raw.min():.4f} – {T_abaqus_raw.max():.4f}")

# Interpolate Abaqus field onto FEniCSx node positions
interp = LinearNDInterpolator(m2.points, T_abaqus_raw)
T_abaqus_on_fenics = interp(points_fenics)

# Nodes outside Abaqus mesh convex hull get NaN — report and mask
n_nan = np.isnan(T_abaqus_on_fenics).sum()
if n_nan:
    print(f"Warning: {n_nan} FEniCSx nodes outside Abaqus mesh hull — excluded from MSE")

mask = ~np.isnan(T_abaqus_on_fenics)
residual = - (T_fenics[mask] - T_abaqus_on_fenics[mask])

print(f"\nResidual range : {residual.min():.4e}  →  {residual.max():.4e}")
print(f"MSE  : {np.mean(residual**2):.3e} °C²")
print(f"RMSE : {np.sqrt(np.mean(residual**2)):.3e} °C")
print(f"MAE  : {np.mean(np.abs(residual)):.3e} °C")

FEniCSx nodes : (111450, 3)  T range: 49.4345 – 49.9983
Abaqus  nodes : (111450, 3)       T range: 49.4374 – 49.9976

Residual range : -7.0060e-03  →  1.2436e-02
MSE  : 1.655e-06 °C²
RMSE : 1.287e-03 °C
MAE  : 9.844e-04 °C
